# BƯỚC 8 — Evaluation & Fairness (Đồ án 5)
**Đánh giá cuối trên tập TEST đã khóa** — sử dụng mô hình đã huấn luyện Bước 7, KHÔNG fit lại, KHÔNG chọn lại mô hình bằng test.

- Mô hình đã chọn từ validation: **DecisionTree** (Recall@10% validation cao hơn RandomForest).
- Đánh giá 3 mô hình trên 15.219 lượt nhập viện của test theo ngưỡng cố định 0.5 và Recall@10%.
- Kiểm tra fairness theo `gender`, `race`, `age`: recall, tỷ lệ cảnh báo, precision theo nhóm. Các nhóm nhỏ cần cảnh báo, không kết luận chắc chắn.
- Phân tích kiểm soát sai lệch không đồng nghĩa chứng minh hiệu quả lâm sàng. Không dùng các mô hình để quyết định chăm sóc thực tế.


In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
from scipy import sparse
import matplotlib.pyplot as plt
from sklearn.metrics import (precision_score, recall_score, f1_score, balanced_accuracy_score,
    average_precision_score, roc_auc_score, confusion_matrix, precision_recall_curve,
    ConfusionMatrixDisplay)
import joblib
from IPython.display import display

RANDOM_STATE=42
cwd=Path.cwd()
ROOT=next((p for p in (cwd,*cwd.parents,Path(r'D:\Readmission_Project')) if (p/'data'/'processed'/'step4'/'X_test.npz').exists()),None)
if ROOT is None: raise FileNotFoundError('Không tìm thấy X_test.npz — mở Jupyter từ thư mục dự án.')
DATA=ROOT/'data'/'processed'/'step4'
TRAIN_REPORT=ROOT/'reports'/'classification'
REPORT=ROOT/'reports'/'evaluation_fairness'
REPORT.mkdir(parents=True,exist_ok=True)
print('ROOT:',ROOT)
print('REPORT:',REPORT)

## 1. Khóa quyết định từ validation
Đọc lựa chọn đã ghi ở Bước 7 và đối chiếu bảng Recall@10%. Nếu không phải `DecisionTree`, dừng để kiểm tra trước khi mở test. Không được đổi tiêu chí/model sau khi nhìn test.

In [ ]:
validation_summary=json.loads((TRAIN_REPORT/'training_summary.json').read_text(encoding='utf-8'))
cap=pd.read_csv(TRAIN_REPORT/'validation_capacity_top10pct.csv')
display(cap)
selected=validation_summary['selected_on_validation']
print('Mô hình khóa theo validation:',selected)
assert selected=='DecisionTree', f'Lựa chọn trước test là {selected}; phải kiểm tra lý do khác DecisionTree.'
assert abs(float(validation_summary['top_fraction'])-0.10)<1e-9
assert validation_summary['test_used'] is False
assert cap.iloc[0]['model']==selected
print('Đã khóa: DecisionTree, top 10%, threshold minh họa 0.5. Không dùng test để tối ưu.')

## 2. Đọc test và kiểm tra ranh giới bệnh nhân
Chỉ đọc test **sau khi** lựa chọn mô hình đã được khóa. Xác nhận tất cả nhãn, ID và tập bệnh nhân không chồng lắp.

In [ ]:
X_test=sparse.load_npz(DATA/'X_test.npz').tocsr()
y_test=np.load(DATA/'y_test.npy').astype(int)
test_ids=pd.read_csv(DATA/'test_ids_labels.csv')
train_ids=pd.read_csv(DATA/'train_ids_labels.csv')
val_ids=pd.read_csv(DATA/'validation_ids_labels.csv')
assert X_test.shape[0]==len(y_test)==len(test_ids)==15219
assert np.array_equal(y_test,test_ids['readmitted_30d'].to_numpy())
assert X_test.shape[1]==int(validation_summary['n_features'])
assert test_ids['encounter_id'].is_unique
assert set(test_ids.patient_nbr).isdisjoint(train_ids.patient_nbr)
assert set(test_ids.patient_nbr).isdisjoint(val_ids.patient_nbr)
assert set(np.unique(y_test)).issubset({0,1})
print('TEST:',X_test.shape,'| Cases:',int(y_test.sum()),'| Prevalence:',round(y_test.mean()*100,2),'%')

## 3. Đánh giá ba mô hình — ngưỡng 0.5 được cố định từ trước
Baseline majority dự đoán mọi lượt thuộc lớp 0. Average Precision của baseline = tỷ lệ dương tính trên test (do score đồng hạng).

In [ ]:
NAMES=['Baseline_majority','DecisionTree','RandomForest']
models={name:joblib.load(TRAIN_REPORT/f'model_{name}.joblib') for name in NAMES}
probs={}; preds={}; metrics=[]
for name,model in models.items():
    assert hasattr(model,'predict_proba')
    prob=model.predict_proba(X_test)[:,list(model.classes_).index(1)]
    pred=(prob>=0.5).astype(int)
    probs[name]=prob; preds[name]=pred
    tn,fp,fn,tp=confusion_matrix(y_test,pred,labels=[0,1]).ravel()
    metrics.append({'model':name,'threshold':0.5,'n_test':len(y_test),
        'precision':precision_score(y_test,pred,zero_division=0),
        'recall':recall_score(y_test,pred,zero_division=0),
        'f1':f1_score(y_test,pred,zero_division=0),
        'balanced_accuracy':balanced_accuracy_score(y_test,pred),
        'average_precision':average_precision_score(y_test,prob),
        'roc_auc':roc_auc_score(y_test,prob),
        'TP':int(tp),'FP':int(fp),'TN':int(tn),'FN':int(fn)})
metric_df=pd.DataFrame(metrics)
display(metric_df.round(4))
metric_df.to_csv(REPORT/'test_metrics_threshold_050.csv',index=False,encoding='utf-8-sig')

## 4. Đánh giá theo năng lực gọi 10% (Recall@10%)
Dùng **top k** theo điểm số giảm dần, không đặt ngưỡng xác suất mới từ test. Khi điểm số bằng nhau, thứ tự dữ liệu làm tie-break và kết quả baseline top10 có thể ngẫu nhiên theo thứ tự dữ liệu; **không coi baseline top10 là một thứ hạng có ý nghĩa**.

In [ ]:
fraction=0.10
k=int(np.ceil(len(y_test)*fraction))
capacity=[]; selected_masks={}
for name,score in probs.items():
    order=np.argsort(-score,kind='stable')
    chosen=order[:k]
    mask=np.zeros(len(y_test),dtype=bool);mask[chosen]=True
    selected_masks[name]=mask
    hits=int(y_test[mask].sum())
    capacity.append({'model':name,'top_fraction':fraction,'capacity_calls':k,
        'true_readmissions_captured':hits,'precision_at_10pct':hits/k,
        'recall_at_10pct':hits/int(y_test.sum()),
        'lift_at_10pct':(hits/k)/y_test.mean()})
cap_test=pd.DataFrame(capacity)
display(cap_test.round(4))
cap_test.to_csv(REPORT/'test_capacity_top10pct.csv',index=False,encoding='utf-8-sig')
print('Kết quả ngẫu nhiên kỳ vọng nếu gọi 10%:',round(k*y_test.mean(),1),'trường hợp; Recall khoảng 10%.')

## 5. Đồ thị Precision–Recall và Confusion Matrices (Test)
Chỉ tạo hình, không điều chỉnh mô hình theo hình.

In [ ]:
fig,ax=plt.subplots(figsize=(8,5))
for name,score in probs.items():
    pr,re,_=precision_recall_curve(y_test,score)
    ax.plot(re,pr,label=f'{name} (AP={average_precision_score(y_test,score):.3f})')
ax.axhline(y_test.mean(),linestyle='--',label=f'Prevalence {y_test.mean():.3f}')
ax.set(xlabel='Recall',ylabel='Precision',title='Precision–Recall trên TEST')
ax.legend();fig.tight_layout()
fig.savefig(REPORT/'01_precision_recall_test.png',dpi=160);plt.show()

fig,axes=plt.subplots(1,len(NAMES),figsize=(13,4))
for ax,name in zip(axes,NAMES):
    ConfusionMatrixDisplay(confusion_matrix(y_test,preds[name],labels=[0,1]),
        display_labels=['No <30','<30']).plot(ax=ax,colorbar=False)
    ax.set_title(name)
fig.tight_layout();fig.savefig(REPORT/'02_confusion_matrices_test.png',dpi=160);plt.show()

## 6. Liên kết thuộc tính nhạy cảm theo encounter_id
Dùng `step3_cleaned.csv` để bổ sung `gender`, `race`, `age` vào các lượt test. Liên kết theo encounter_id; tuyệt đối không ghép theo chỉ số dòng sau khi chia.

In [ ]:
source=ROOT/'data'/'processed'/'step3_cleaned.csv'
sens=pd.read_csv(source,usecols=['encounter_id','patient_nbr','gender','race','age'],low_memory=False)
assert sens.encounter_id.is_unique
analysis=test_ids[['encounter_id','patient_nbr','readmitted_30d']].merge(
    sens,on=['encounter_id','patient_nbr'],how='left',validate='one_to_one',sort=False,indicator=True)
assert len(analysis)==len(y_test)
assert (analysis['_merge']=='both').all()
assert np.array_equal(analysis.readmitted_30d.to_numpy(),y_test)
analysis=analysis.drop(columns=['_merge'])
for col in ['gender','race','age']:
    analysis[col]=analysis[col].astype('string').fillna('Unknown').replace({'?':'Unknown','':'Unknown'})
analysis['score']=probs[selected]
analysis['pred_050']=preds[selected]
analysis['selected_top10']=selected_masks[selected].astype(int)
print('Fairness basis:',len(analysis),'encounters (matched by encounter_id)')
print('Số nhóm: ',{c:analysis[c].nunique() for c in ['gender','race','age']})

## 7. Chỉ số fairness theo nhóm
So sánh **Recall/TPR** tại ngưỡng 0.5, **FPR**, precision, tỷ lệ được cảnh báo và tỷ lệ được chọn top10% toàn bộ test. Nhóm có <100 lượt hoặc <30 ca dương tính được gắn cờ **không đủ cỡ mẫu để diễn giải chắc chắn**. Những thống kê này đánh giá theo lượt khám, không phải theo số người; không suy ra công bằng lâm sàng.

In [ ]:
def group_metrics(data,column):
    rows=[]
    for value,g in data.groupby(column,dropna=False,sort=True):
        yt=g.readmitted_30d.to_numpy(dtype=int)
        yp=g.pred_050.to_numpy(dtype=int)
        tn,fp,fn,tp=confusion_matrix(yt,yp,labels=[0,1]).ravel()
        n=len(g);pos=int(yt.sum());neg=n-pos
        rows.append({'attribute':column,'group':str(value),'n':n,'positives':pos,'negatives':neg,
            'prevalence':pos/n,'TP':int(tp),'FP':int(fp),'TN':int(tn),'FN':int(fn),
            'recall_tpr':tp/pos if pos else np.nan,
            'fpr':fp/neg if neg else np.nan,
            'precision':tp/(tp+fp) if (tp+fp) else np.nan,
            'flag_rate_050':float(yp.mean()),
            'selection_rate_top10':float(g.selected_top10.mean()),
            'recall_of_positives_top10':float(g.loc[g.readmitted_30d==1,'selected_top10'].mean()) if pos else np.nan,
            'interpretation_warning':'SMALL_GROUP' if n<100 or pos<30 else 'OK'})
    return pd.DataFrame(rows)
fairness=pd.concat([group_metrics(analysis,c) for c in ['gender','race','age']],ignore_index=True)
display(fairness.round(4))
fairness.to_csv(REPORT/'test_fairness_by_group.csv',index=False,encoding='utf-8-sig')
print('NHÓM NHỎ — không diễn giải kết luận:')
display(fairness.loc[fairness.interpretation_warning=='SMALL_GROUP',['attribute','group','n','positives']])

## 8. Khoảng chênh lệch Recall theo nhóm đủ số lượng
Chỉ là mô tả theo mẫu; **không phải** kiểm định ý nghĩa thống kê. Không dùng kết quả trên test để điều chỉnh mô hình hoặc ngưỡng rồi báo cáo như đánh giá độc lập.

In [ ]:
fairness_gap=[]
for attribute,grp in fairness.groupby('attribute'):
    eligible=grp[(grp.interpretation_warning=='OK') & grp.recall_tpr.notna()]
    fairness_gap.append({'attribute':attribute,'groups_eligible':int(len(eligible)),
        'min_recall':eligible.recall_tpr.min() if len(eligible) else np.nan,
        'max_recall':eligible.recall_tpr.max() if len(eligible) else np.nan,
        'recall_gap_pp':100*(eligible.recall_tpr.max()-eligible.recall_tpr.min()) if len(eligible)>1 else np.nan})
gaps=pd.DataFrame(fairness_gap)
display(gaps.round(3))
gaps.to_csv(REPORT/'test_fairness_recall_gaps.csv',index=False,encoding='utf-8-sig')
fig,ax=plt.subplots(figsize=(9,4))
sub=fairness[(fairness.attribute=='race')&(fairness.interpretation_warning=='OK')].copy()
ax.bar(sub.group.astype(str),sub.recall_tpr.astype(float))
ax.set(ylabel='Recall / TPR',title='Recall theo race (nhóm đủ số lượng)',ylim=(0,1))
ax.tick_params(axis='x',rotation=25)
fig.tight_layout();fig.savefig(REPORT/'03_fairness_recall_race.png',dpi=160);plt.show()

## 9. Ghi nhật ký quyết định, bảng tổng kết và lưu kết quả
Không lưu danh sách bệnh nhân ưu tiên ra báo cáo chia sẻ. Chỉ ghi bảng thống kê tổng hợp; đây là dữ liệu lượt khám có yếu tố nhạy cảm.

In [ ]:
decision_log=pd.DataFrame([
['D08-01','Giữ DecisionTree đã chọn bằng Recall@10% trên validation','Tránh lựa chọn mô hình dựa trên test','Recall@10% test là đánh giá cuối'],
['D08-02','Ngưỡng 0.5 được cố định từ trước; top 10% theo thứ hạng score','Giới hạn nhân lực gọi theo dõi 10%','Confusion matrix và Recall@10%'],
['D08-03','Đối chiếu Baseline_majority và RandomForest','Chứng minh giá trị tăng thêm so với nền và mô hình đối chứng','Precision/Recall/AP/ROC-AUC'],
['D08-04','Fairness theo gender, race và age dựa encounter_id','Đánh giá nguy cơ bỏ sót không đồng đều','Recall/TPR, FPR, top10 selection'],
['D08-05','Cảnh báo nhóm dưới 100 lượt hoặc dưới 30 ca dương','Nhóm quá nhỏ dễ có tỷ lệ không ổn định','Giới hạn kết luận fairness'],
['D08-06','Không điều chỉnh theo test; giữ nguyên test cho báo cáo cuối','Tránh leakage do điều chỉnh hậu kiểm','Độ tin cậy kết luận'],
['D08-07','Không xuất danh sách người bệnh ra thư mục báo cáo','Giảm rủi ro quyền riêng tư','Dữ liệu chỉ gồm tổng hợp']
],columns=['ma','quyet_dinh','ly_do','anh_huong'])
decision_log.to_csv(REPORT/'decision_log_step8.csv',index=False,encoding='utf-8-sig')
summary={'locked_model':selected,'selection_metric':'recall_at_10pct_validation',
 'test_rows':int(len(y_test)),'test_positives':int(y_test.sum()),'test_prevalence':float(y_test.mean()),
 'capacity_calls':int(k),'model_ever_refit_on_test':False,
 'patient_disjoint_from_train_validation':True,'fairness_attributes':['gender','race','age'],
 'random_state':RANDOM_STATE}
(REPORT/'test_evaluation_summary.json').write_text(json.dumps(summary,ensure_ascii=False,indent=2),encoding='utf-8')
print('Đã lưu kết quả tại:',REPORT)
print('TEST:',X_test.shape,'| positive:',int(y_test.sum()))
print('MODEL ĐÃ KHÓA:',selected)
print('Recall@10%:',round(float(cap_test.set_index('model').loc[selected,'recall_at_10pct']),4))
print('HOÀN THÀNH BƯỚC 8 (đánh giá test một lần, fairness mô tả).')

## Lưu ý khi viết báo cáo
- Đây là kết quả trên dữ liệu lịch sử, không phải thử nghiệm bệnh viện thực tế; mô hình không thay thế chuyên gia y tế.
- `race`, `gender`, `age` là nhóm cần được quan sát fairness, không được diễn giải chênh lệch như quan hệ nhân quả.
- Kiểm tra thêm nhóm bệnh nhân nhỏ và các giá trị thiếu, độ bất định thống kê nếu đưa vào triển khai.
- Nếu muốn điều chỉnh mô hình sau khi xem test, cần thiết lập một tập kiểm tra mới độc lập cho đánh giá xác nhận tiếp theo.
